# SimTrust-Pre — Stage 1: Complete data-feasibility audit (v4.2)

**Purpose:** audit the public batch-distillation dataset and freeze the evidence needed before testing whether pre-perturbation simulation fidelity predicts the training utility of simulated fault trajectories.

**This notebook is an audit notebook, not a model-training notebook.** It does not automatically certify run pairing, physical variable mappings, perturbation time, uncertainty semantics, or scientific novelty.

## What this notebook does

1. Pins Zenodo record `22250958`, version `1.1.3`, DOI `10.5281/zenodo.22250958` and checks files before downloading.
2. Downloads only metadata and measured time-series archives needed for feasibility checks. The approximately 2.1 GB simulation time-series archive stays disabled by default.
3. Audits experiment/phase paths, file overlap, schema samples, uncertainty-column dynamics, interpolation flags, YAML anomaly metadata, and every XML simulation configuration.
4. Writes a populated **candidate** perturbation registry and a human-review ledger. `PerturbationMode.hasBeginning` is a candidate boundary until its unit, time origin, and mapping to the real experiment are reviewed.
5. Provides tested, illustrative functions for measurement-tolerance-scaled discrepancy in pre-onset windows. They are not calibrated probabilities and are not applied to dataset data automatically.
6. Saves CSV/JSON/TXT artifacts in `WORK_DIR` for review and sharing.

## Scientific guardrails

- `Data_interpolated` is counted separately as true, false, missing, or unrecognized. Its meaning must still be confirmed from documentation/source before using it to filter rows.
- A sensor-uncertainty/tolerance column is **not automatically** a standard deviation or 95% confidence interval. A zero can mean unavailable; it must not be used as a tiny denominator.
- Do not equate `PerturbationMode.hasBeginning` with `anomalies.hasBeginning`: an interval can exist between applying a perturbation and observing an anomaly.
- Only simulator-covered anomaly types may be included in paired simulation-to-real claims.
- Similar path IDs are structural candidates, not proof that two records are scientifically paired.
- No dataset result or novelty claim is confirmed until the generated evidence files are reviewed.

**Run cells from top to bottom in Google Colab.** If a download or archive scan errors, keep the full traceback and the saved error CSV/JSON files.

In [ ]:
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
import os
import re
import json
import hashlib
import zipfile
import xml.etree.ElementTree as ET
from urllib.parse import quote

import numpy as np
import pandas as pd
import requests
from IPython.display import display

try:
    import yaml
except ImportError as exc:
    raise ImportError(
        "PyYAML is required for YAML metadata. In Colab, run `%pip -q install pyyaml`, "
        "then rerun this cell."
    ) from exc

RECORD_ID = "22250958"
EXPECTED_VERSION = "1.1.3"
RECORD_DOI = "10.5281/zenodo.22250958"
API_URL = f"https://zenodo.org/api/records/{RECORD_ID}"
WORK_DIR = Path("/content/simtrust_pre_audit") if Path("/content").exists() else Path("simtrust_pre_audit")
WORK_DIR.mkdir(parents=True, exist_ok=True)

LABEL_KEY = "00_Batch_Distillation_Plant_M-202210_Timeseries_Label_Anomaly_Metadata.zip"
SENSOR_KEY = "01_Batch_Distillation_Plant_M-202210_Timeseries_Sensors.zip"
ACTUATOR_KEY = "02_Batch_Distillation_Plant_M-202210_Timeseries_Actuators.zip"
UNC_KEY = "03_Batch_Distillation_Plant_M-202210_Timeseries_Sensor_Uncertainties.zip"
CONFIG_KEY = "14_Batch_Distillation_Plant_M-202210_Timeseries_Simulation_Configuration.zip"
AVAILABILITY_KEY = "Batch_Distillation_Plant_M-202210_Modality_Availability_Matrix.csv"
AVAILABILITY_OVERVIEW_KEY = "Batch_Distillation_Plant_M-202210_Modality_Availability_Overview.md"
SIM_KEY = "13_Batch_Distillation_Plant_M-202210_Timeseries_Simulation.zip"

EXPECTED_SMALL_FILES = [
    LABEL_KEY, SENSOR_KEY, ACTUATOR_KEY, UNC_KEY, CONFIG_KEY,
    AVAILABILITY_KEY, AVAILABILITY_OVERVIEW_KEY,
]
PHASES = {"Startup", "Operation", "Shutdown"}

print("Working directory:", WORK_DIR.resolve())
print("Notebook purpose: Stage 1 audit only; no detector training.")

## 1. Verify the pinned Zenodo record and download the small audit files

This step checks the version and DOI returned by Zenodo before downloading. Each file is verified against the record's reported size and MD5 checksum when available. It never extracts ZIP files to disk.

In [ ]:
response = requests.get(API_URL, timeout=60)
response.raise_for_status()
record = response.json()
metadata = record.get("metadata", {})
record_files = record.get("files", [])

api_version = str(metadata.get("version") or "").strip()
api_doi = str(record.get("doi") or metadata.get("doi") or "").strip()
if not api_version:
    raise RuntimeError("Zenodo API did not report a version. Stop and verify the pinned record manually.")
if api_version != EXPECTED_VERSION:
    raise RuntimeError(f"Version mismatch: expected {EXPECTED_VERSION}, received {api_version}.")
if api_doi and api_doi.lower() != RECORD_DOI.lower():
    raise RuntimeError(f"DOI mismatch: expected {RECORD_DOI}, received {api_doi}.")
if not record_files:
    raise RuntimeError("Zenodo API returned no file metadata.")

file_by_key = {f.get("key"): f for f in record_files if f.get("key")}
manifest_rows = []
for f in record_files:
    manifest_rows.append({
        "key": f.get("key", ""),
        "size_bytes": int(f.get("size") or 0),
        "size_MB": round((f.get("size") or 0) / 1_000_000, 3),
        "size_GB": round((f.get("size") or 0) / 1_000_000_000, 4),
        "checksum": f.get("checksum", ""),
    })
manifest_df = pd.DataFrame(manifest_rows).sort_values("size_bytes")
manifest_df.to_csv(WORK_DIR / "record_file_manifest.csv", index=False)

print("Record title:", metadata.get("title"))
print("Version:", api_version, "| DOI:", api_doi or "NOT PROVIDED")
print("Pinned version/DOI check: PASS")
print("License metadata:", metadata.get("license"))
display(manifest_df)

missing_files = [key for key in EXPECTED_SMALL_FILES if key not in file_by_key]
if missing_files:
    print("\nExpected audit files absent from this record response:")
    for key in missing_files:
        print(" -", key)
    print("Available related files:")
    for key in file_by_key:
        if any(token in key.lower() for token in ["anomaly", "sensor", "actuator", "simulation", "availability"]):
            print(" -", key)
    raise RuntimeError("At least one expected audit file is absent. Inspect the printed record manifest before continuing.")

print("\nSaved manifest:", WORK_DIR / "record_file_manifest.csv")

In [ ]:
def get_file_url(file_meta):
    links = file_meta.get("links", {}) or {}
    for name in ("download", "content", "self"):
        if links.get(name):
            url = links[name]
            if name == "self" and "download=" not in url:
                return url + ("&" if "?" in url else "?") + "download=1"
            return url
    return f"https://zenodo.org/records/{RECORD_ID}/files/{quote(file_meta['key'])}?download=1"


def local_file_matches(path, file_meta):
    path = Path(path)
    expected_size = file_meta.get("size")
    if expected_size is not None and path.stat().st_size != int(expected_size):
        return False
    expected_checksum = str(file_meta.get("checksum") or "")
    if expected_checksum.startswith("md5:"):
        digest = hashlib.md5()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest().lower() == expected_checksum.split(":", 1)[1].lower()
    return True


def download_file(key, chunk_size=1024 * 1024):
    if key not in file_by_key:
        raise KeyError(f"File not present in the pinned Zenodo record: {key}")
    dest = WORK_DIR / key
    meta = file_by_key[key]
    if dest.exists():
        if local_file_matches(dest, meta):
            print("Verified existing download:", key)
            return dest
        print("Existing file did not match record size/checksum; redownloading:", key)
        dest.unlink()

    expected_size = int(meta.get("size") or 0)
    print(f"Downloading {key} ({expected_size / 1_000_000:.2f} MB)")
    tmp = dest.with_suffix(dest.suffix + ".part")
    try:
        with requests.get(get_file_url(meta), stream=True, timeout=(30, 180), allow_redirects=True) as r:
            r.raise_for_status()
            with tmp.open("wb") as out:
                for chunk in r.iter_content(chunk_size=chunk_size):
                    if chunk:
                        out.write(chunk)
        if not local_file_matches(tmp, meta):
            raise IOError(f"Downloaded file failed size/checksum validation: {key}")
        os.replace(tmp, dest)
    except Exception:
        if tmp.exists():
            tmp.unlink()
        raise
    return dest


local_files = {}
for key in EXPECTED_SMALL_FILES:
    local_files[key] = download_file(key)

print("\nDownloaded/verified audit files:")
for key, path in local_files.items():
    print(f" - {key}: {Path(path).stat().st_size:,} bytes")

## 2. Path identity, archive inventory, and structural run matching

The canonical identity is built from the phase, chemical system, operating point, and experiment. XML configurations may use either `.../experiment/config.xml` or `.../experiment.xml`, and may omit the phase directory. Those no-phase XML records are assigned `Operation` **only as a structural candidate**; that does not prove a scientific match.

In [ ]:
def identity_from_member(member):
    """Return (phase, canonical_run_id) from a ZIP member path, when possible.

    Phase-less YAML/XML paths are structural candidates only. Their assumed phase is
    Operation and must be verified manually before treating them as paired records.
    Supports both <experiment>.yaml and <experiment>/metadata.yaml layouts.
    """
    pth = PurePosixPath(str(member))
    parts = list(pth.parts)
    phase_idx = next((i for i, part in enumerate(parts) if part in PHASES), None)
    metadata_filenames = {"config.xml", "metadata.yaml", "metadata.yml"}

    if phase_idx is not None:
        if len(parts) <= phase_idx + 2:
            return None
        phase = parts[phase_idx]
        chemical = parts[phase_idx + 1]
        operating_point = parts[phase_idx + 2]
        if pth.name.lower() in metadata_filenames:
            # <phase>/<chemical>/<operating_point>/<experiment>/<metadata-file>
            if len(parts) <= phase_idx + 3:
                return None
            experiment = parts[phase_idx + 3]
        else:
            # <phase>/<chemical>/<operating_point>/<experiment>.<ext>
            experiment = pth.stem
    elif pth.suffix.lower() in {".xml", ".yaml", ".yml"}:
        # Phase-less layouts are supported as candidate identities only.
        if pth.name.lower() in metadata_filenames:
            # <chemical>/<operating_point>/<experiment>/<metadata-file>
            experiment = pth.parent.name
            operating_point = pth.parent.parent.name
            chemical = pth.parent.parent.parent.name
        else:
            # <chemical>/<operating_point>/<experiment>.<ext>
            experiment = pth.stem
            operating_point = pth.parent.name
            chemical = pth.parent.parent.name
        if not all((chemical, operating_point, experiment)):
            return None
        phase = "Operation"  # structural assumption only; review required
    else:
        return None

    canonical = f"{chemical}/{operating_point}/{experiment}"
    return phase, canonical


# Offline path-parser regression tests. They validate path parsing only, not real pairing.
_identity_cases = {
    "archive/Operation/chem_A/OP_01/experiment_001/config.xml": ("Operation", "chem_A/OP_01/experiment_001"),
    "archive/chem_A/OP_01/experiment_001/config.xml": ("Operation", "chem_A/OP_01/experiment_001"),
    "archive/Operation/chem_A/OP_01/experiment_001.xml": ("Operation", "chem_A/OP_01/experiment_001"),
    "archive/chem_A/OP_01/experiment_001.xml": ("Operation", "chem_A/OP_01/experiment_001"),
    "archive/Operation/chem_A/OP_01/experiment_001.yaml": ("Operation", "chem_A/OP_01/experiment_001"),
    "archive/Operation/chem_A/OP_01/experiment_001.yml": ("Operation", "chem_A/OP_01/experiment_001"),
    "archive/Operation/chem_A/OP_01/experiment_001/metadata.yaml": ("Operation", "chem_A/OP_01/experiment_001"),
    "archive/Operation/chem_A/OP_01/experiment_001/metadata.yml": ("Operation", "chem_A/OP_01/experiment_001"),
    "archive/chem_A/OP_01/experiment_001.yaml": ("Operation", "chem_A/OP_01/experiment_001"),
    "archive/chem_A/OP_01/experiment_001.yml": ("Operation", "chem_A/OP_01/experiment_001"),
    "archive/chem_A/OP_01/experiment_001/metadata.yaml": ("Operation", "chem_A/OP_01/experiment_001"),
    "archive/chem_A/OP_01/experiment_001/metadata.yml": ("Operation", "chem_A/OP_01/experiment_001"),
    "archive/Operation/chem_A/OP_01/experiment_001.csv": ("Operation", "chem_A/OP_01/experiment_001"),
    "archive/Shutdown/chem_A/OP_01/experiment_001.csv": ("Shutdown", "chem_A/OP_01/experiment_001"),
}
for _member, _expected in _identity_cases.items():
    _actual = identity_from_member(_member)
    assert _actual == _expected, f"Path parser failed: {_member!r}: {_actual!r} != {_expected!r}"
print(f"Path-parser synthetic tests passed: {len(_identity_cases)} cases.")


def list_members(path):
    with zipfile.ZipFile(path) as zf:
        return [info for info in zf.infolist() if not info.is_dir()]

archive_inventory_rows = []
archive_index = {}
archive_counts = {}
for key, path in local_files.items():
    if str(path).lower().endswith(".zip"):
        infos = list_members(path)
        counts = Counter(PurePosixPath(info.filename).suffix.lower() for info in infos)
        identities = set()
        member_identity_rows = []
        for info in infos:
            ident = identity_from_member(info.filename)
            if ident:
                identities.add(ident)
                member_identity_rows.append({
                    "archive": key, "member": info.filename,
                    "phase": ident[0], "canonical_run_id": ident[1],
                    "uncompressed_bytes": info.file_size,
                })
            archive_inventory_rows.append({
                "archive": key, "member": info.filename,
                "suffix": PurePosixPath(info.filename).suffix.lower(),
                "compressed_bytes": info.compress_size,
                "uncompressed_bytes": info.file_size,
                "phase_candidate": next((p for p in PurePosixPath(info.filename).parts if p in PHASES), "UNKNOWN"),
            })
        archive_counts[key] = {"members": len(infos), **dict(counts)}
        archive_index[key] = {
            "members": [i.filename for i in infos],
            "identities": identities,
            "csv": [i.filename for i in infos if i.filename.lower().endswith(".csv")],
            "yaml": [i.filename for i in infos if i.filename.lower().endswith((".yaml", ".yml"))],
            "xml": [i.filename for i in infos if i.filename.lower().endswith(".xml")],
            "identity_rows": member_identity_rows,
        }

archive_inventory_df = pd.DataFrame(archive_inventory_rows)
archive_inventory_path = WORK_DIR / "archive_member_inventory.csv"
archive_inventory_df.to_csv(archive_inventory_path, index=False)
print("\nArchive inventory saved:", archive_inventory_path)
for key, stats in archive_counts.items():
    print(f"\n[{key}] {json.dumps(stats, ensure_ascii=False, sort_keys=True)}")

for a, b in [(LABEL_KEY, SENSOR_KEY), (LABEL_KEY, UNC_KEY), (LABEL_KEY, CONFIG_KEY), (SENSOR_KEY, UNC_KEY)]:
    common = archive_index[a]["identities"] & archive_index[b]["identities"]
    print(f"\nShared phase/run identities: {Path(a).name} <> {Path(b).name}: {len(common)}")
    for identity in sorted(common)[:4]:
        print("  ", identity[0], "/", identity[1])

base_keys = [LABEL_KEY, SENSOR_KEY, UNC_KEY, CONFIG_KEY]
all_four_shared = set.intersection(*(archive_index[k]["identities"] for k in base_keys))
print("\nAll-four-archive shared phase/run identities:", len(all_four_shared))
for phase in sorted(PHASES):
    print(f"  {phase}: {sum(1 for p, _ in all_four_shared if p == phase)}")

# Full experiment_path_audit across all five ZIP archives, with explicit membership flags.
audit_archive_keys = [LABEL_KEY, SENSOR_KEY, ACTUATOR_KEY, UNC_KEY, CONFIG_KEY]
all_structural_identities = set().union(*(archive_index[k]["identities"] for k in audit_archive_keys))
path_audit_rows = []
for phase, canonical_run_id in sorted(all_structural_identities):
    row = {"phase": phase, "canonical_run_id": canonical_run_id}
    for key in audit_archive_keys:
        safe_key = {
            LABEL_KEY: "has_label_metadata",
            SENSOR_KEY: "has_sensor_csv",
            ACTUATOR_KEY: "has_actuator_csv",
            UNC_KEY: "has_uncertainty_csv",
            CONFIG_KEY: "has_config_xml",
        }[key]
        row[safe_key] = (phase, canonical_run_id) in archive_index[key]["identities"]
    row["present_in_all_five"] = all(row[k] for k in [
        "has_label_metadata", "has_sensor_csv", "has_actuator_csv",
        "has_uncertainty_csv", "has_config_xml"
    ])
    path_audit_rows.append(row)
path_audit_df = pd.DataFrame(path_audit_rows)
path_audit_df.to_csv(WORK_DIR / "experiment_path_audit.csv", index=False)
print("Saved full structural experiment-path audit:", WORK_DIR / "experiment_path_audit.csv")
if not path_audit_df.empty:
    print("Structural identities present in all five ZIP archives:", int(path_audit_df["present_in_all_five"].sum()))
    display(path_audit_df.head(20))

# Inspect the dataset's modality availability matrix without using it as proof of run pairing.
if AVAILABILITY_KEY in local_files:
    availability_matrix = pd.read_csv(local_files[AVAILABILITY_KEY])
    print("\nModality availability matrix shape:", availability_matrix.shape)
    display(availability_matrix.head(10))
    availability_matrix.to_csv(WORK_DIR / "modality_availability_matrix_copy.csv", index=False)

pd.DataFrame([
    {"archive_a": a, "archive_b": b,
     "shared_phase_run_identities": len(archive_index[a]["identities"] & archive_index[b]["identities"])}
    for a, b in [(LABEL_KEY, SENSOR_KEY), (LABEL_KEY, UNC_KEY), (LABEL_KEY, CONFIG_KEY), (SENSOR_KEY, UNC_KEY)]
]).to_csv(WORK_DIR / "shared_path_overlap_summary.csv", index=False)

## 3. Schema samples and label CSV audit

Schema samples are drawn across available phase/chemical groups, rather than only from the first few ZIP entries. Label CSV values are summarized to reveal label fields and compact categorical values; this does not decide whether a label is binary or which classes should be merged.

In [ ]:
schema_rows = []
label_column_counts = Counter()
label_values = defaultdict(set)
label_read_errors = []
label_hint = re.compile(r"label|anomal|phase|fault|state|interpol|perturb|event|status", re.I)

for key, path in local_files.items():
    if not str(path).lower().endswith(".zip"):
        continue
    with zipfile.ZipFile(path) as zf:
        csv_members = [n for n in zf.namelist() if n.lower().endswith(".csv") and not n.endswith("/")]
        # Pick up to two schema examples per (phase, chemical) group, plus early fallback examples.
        chosen = []
        seen_groups = set()
        for member in csv_members:
            ident = identity_from_member(member)
            if ident:
                phase, canonical = ident
                chemical = canonical.split("/")[0]
                group = (phase, chemical)
                if group not in seen_groups:
                    chosen.append(member)
                    seen_groups.add(group)
            if len(chosen) >= 12:
                break
        for member in chosen or csv_members[:3]:
            try:
                with zf.open(member) as stream:
                    head = pd.read_csv(stream, nrows=2, low_memory=False)
                schema_rows.append({
                    "archive": key, "member": member,
                    "columns_count": len(head.columns),
                    "columns_json": json.dumps([str(c) for c in head.columns], ensure_ascii=False),
                    "sample_rows_json": json.dumps(head.head(1).where(pd.notna(head.head(1)), None).to_dict(orient="records"), ensure_ascii=False, default=str),
                })
            except Exception as exc:
                schema_rows.append({"archive": key, "member": member, "error": repr(exc)})

        if key == LABEL_KEY:
            for member in csv_members:
                try:
                    with zf.open(member) as stream:
                        df = pd.read_csv(stream, low_memory=False)
                    for col in df.columns:
                        col_name = str(col)
                        if label_hint.search(col_name) and col_name.lower() not in {"time", "timestamp", "t", "index"}:
                            label_column_counts[col_name] += 1
                            unique_vals = df[col].dropna().astype(str).unique()
                            if len(unique_vals) <= 50:
                                label_values[col_name].update(unique_vals.tolist())
                except Exception as exc:
                    label_read_errors.append({"member": member, "error": repr(exc)})

schema_df = pd.DataFrame(schema_rows)
schema_df.to_csv(WORK_DIR / "csv_schema_samples.csv", index=False)
pd.DataFrame(label_read_errors).to_csv(WORK_DIR / "label_csv_read_errors.csv", index=False)
print("Schema sample rows:", len(schema_df))
print("Saved:", WORK_DIR / "csv_schema_samples.csv")
display(schema_df.head(30))

label_audit_rows = []
for col, count in sorted(label_column_counts.items()):
    label_audit_rows.append({
        "column": col,
        "files_containing_column": int(count),
        "sample_unique_values_json": json.dumps(sorted(label_values.get(col, set()))[:50], ensure_ascii=False),
    })
label_audit_df = pd.DataFrame(label_audit_rows)
label_audit_df.to_csv(WORK_DIR / "label_column_diagnostics.csv", index=False)
print("\nLabel CSV read errors:", len(label_read_errors))
print("Potential label/phase fields:")
display(label_audit_df)
print("Reminder: confirm class semantics in metadata/paper before defining the detection target.")

## 4. Full interpolation-flag audit across uncertainty CSVs

This scans every CSV in the sensor-uncertainty ZIP and calculates per-file fractions only among explicitly recognized boolean values. `True`, `False`, missing values, and unrecognized values stay distinct. The aggregate is a row-level descriptive statistic across files/phases, **not a count of independent experiments**. A column name does not prove its semantics.

In [ ]:
def normalize_flag_values(series):
    """Normalize binary flag values while keeping missing/unknown values separate.

    pandas may parse columns containing blanks as floats, yielding string tokens "1.0"
    and "0.0" after astype("string"); these are accepted as binary values.
    """
    vals = series.astype("string").str.strip().str.lower()
    true_mask = vals.isin(["1", "1.0", "true", "yes", "t", "y"])
    false_mask = vals.isin(["0", "0.0", "false", "no", "f", "n"])
    missing_mask = vals.isna() | vals.isin(["", "nan", "none", "null", "<na>"])
    unknown_mask = ~(true_mask | false_mask | missing_mask)
    return true_mask, false_mask, missing_mask, unknown_mask


# Offline regression test: numeric floats, explicit booleans, missing and unknown tokens.
_flag_test = pd.Series([1.0, 0.0, np.nan, "TRUE", "no", "<NA>", "unexpected"])
_tm, _fm, _mm, _um = normalize_flag_values(_flag_test)
assert int(_tm.sum()) == 2, "Flag parser failed to recognize 1.0/TRUE as true"
assert int(_fm.sum()) == 2, "Flag parser failed to recognize 0.0/no as false"
assert int(_mm.sum()) == 2, "Flag parser failed to recognize NaN/<NA> as missing"
assert int(_um.sum()) == 1, "Flag parser should leave unexpected tokens unrecognized"
print("Interpolation-flag normalization synthetic tests passed: 4 categories.")
del _flag_test, _tm, _fm, _mm, _um


def exact_column_name(columns, wanted):
    wanted = wanted.strip().lower()
    return next((c for c in columns if str(c).strip().lower() == wanted), None)


flag_rows = []
flag_read_errors = []
with zipfile.ZipFile(local_files[UNC_KEY]) as zf:
    uncertainty_csvs = [n for n in zf.namelist() if n.lower().endswith(".csv") and not n.endswith("/")]
    print("Uncertainty CSV members:", len(uncertainty_csvs))
    for member in uncertainty_csvs:
        ident = identity_from_member(member)
        try:
            with zf.open(member) as stream:
                frame = pd.read_csv(stream, low_memory=False)
            common = {
                "member": member,
                "phase": ident[0] if ident else "UNKNOWN",
                "canonical_run_id": ident[1] if ident else None,
                "total_rows": int(len(frame)),
            }
            flag_col = exact_column_name(frame.columns, "Data_interpolated")
            if flag_col is None:
                flag_rows.append({
                    **common, "flag_column_present": False,
                    "interpolated_rows": None, "non_interpolated_rows": None,
                    "missing_flag_values": None, "unrecognized_flag_values": None,
                    "recognized_flag_values": 0, "interpolated_fraction_among_recognized": None,
                })
                continue

            true_mask, false_mask, missing_mask, unknown_mask = normalize_flag_values(frame[flag_col])
            n_true = int(true_mask.sum())
            n_false = int(false_mask.sum())
            n_missing = int(missing_mask.sum())
            n_unknown = int(unknown_mask.sum())
            recognized = n_true + n_false
            flag_rows.append({
                **common, "flag_column_present": True,
                "interpolated_rows": n_true,
                "non_interpolated_rows": n_false,
                "missing_flag_values": n_missing,
                "unrecognized_flag_values": n_unknown,
                "recognized_flag_values": recognized,
                "interpolated_fraction_among_recognized": (n_true / recognized if recognized else np.nan),
            })
        except Exception as exc:
            flag_read_errors.append({"member": member, "error": repr(exc)})

flags_df = pd.DataFrame(flag_rows)
flag_errors_df = pd.DataFrame(flag_read_errors)
flags_df.to_csv(WORK_DIR / "interpolation_flag_full_audit.csv", index=False)
flag_errors_df.to_csv(WORK_DIR / "interpolation_flag_read_errors.csv", index=False)

print("\n=== Full interpolation audit ===")
print("Files successfully audited:", len(flags_df))
print("File read errors:", len(flag_errors_df))
print("Files without Data_interpolated column:", int((~flags_df["flag_column_present"]).sum()) if len(flags_df) else 0)
for col in ["interpolated_rows", "non_interpolated_rows", "missing_flag_values", "unrecognized_flag_values", "recognized_flag_values"]:
    flags_df[col] = pd.to_numeric(flags_df[col], errors="coerce")
true_total = int(flags_df["interpolated_rows"].fillna(0).sum()) if len(flags_df) else 0
false_total = int(flags_df["non_interpolated_rows"].fillna(0).sum()) if len(flags_df) else 0
missing_total = int(flags_df["missing_flag_values"].fillna(0).sum()) if len(flags_df) else 0
unknown_total = int(flags_df["unrecognized_flag_values"].fillna(0).sum()) if len(flags_df) else 0
recognized_total = true_total + false_total
print("Rows across audited files (includes phases/files, not independent experiments):", int(flags_df["total_rows"].sum()) if len(flags_df) else 0)
print("Recognized flag values (True + False):", recognized_total)
print("Flagged interpolated rows:", true_total)
print("Flagged non-interpolated rows:", false_total)
print("Missing flag values:", missing_total)
print("Unrecognized flag values:", unknown_total)
print("Interpolated fraction among recognized flags:", f"{true_total / recognized_total:.6f}" if recognized_total else "UNDEFINED")

print("\nTop 15 files by interpolation fraction among recognized flag values:")
top_flags = (flags_df[flags_df["interpolated_fraction_among_recognized"].notna()]
             .sort_values("interpolated_fraction_among_recognized", ascending=False)
             .head(15))
display(top_flags[["canonical_run_id", "phase", "total_rows", "interpolated_rows",
                   "non_interpolated_rows", "missing_flag_values", "unrecognized_flag_values",
                   "interpolated_fraction_among_recognized"]])

if len(flags_df):
    by_phase = flags_df.groupby("phase", dropna=False)[["interpolated_rows", "non_interpolated_rows", "missing_flag_values", "unrecognized_flag_values", "total_rows"]].sum(min_count=1).reset_index()
    print("\nCounts by phase (descriptive row counts):")
    display(by_phase)
    by_phase.to_csv(WORK_DIR / "interpolation_flag_by_phase.csv", index=False)
print("Saved:", WORK_DIR / "interpolation_flag_full_audit.csv")
print("Saved:", WORK_DIR / "interpolation_flag_read_errors.csv")

## 5. Uncertainty-column dynamics and other interpolation-flag candidates

For every uncertainty CSV and every column, this records value counts, numeric ranges, zero/negative fractions, missingness, and within-file variation. These are diagnostics only: names and numerical variation do not establish a column's physical meaning. The sensor archive is also scanned for column names that suggest interpolation flags.

In [ ]:
uncertainty_column_rows = []
uncertainty_file_rows = []
uncertainty_errors = []
column_name_hint = re.compile(r"uncert|sigma|tolerance|measurement|interpol|data_interpolated|^u(?:_|$)", re.I)
interpolation_name_hint = re.compile(r"interpol", re.I)

with zipfile.ZipFile(local_files[UNC_KEY]) as zf:
    csvs = [n for n in zf.namelist() if n.lower().endswith(".csv") and not n.endswith("/")]
    for member in csvs:
        ident = identity_from_member(member)
        try:
            with zf.open(member) as stream:
                frame = pd.read_csv(stream, low_memory=False)
            uncertainty_file_rows.append({
                "member": member,
                "phase": ident[0] if ident else "UNKNOWN",
                "canonical_run_id": ident[1] if ident else None,
                "n_rows": int(len(frame)),
                "n_columns": int(len(frame.columns)),
                "columns_json": json.dumps([str(c) for c in frame.columns], ensure_ascii=False),
            })
            for col in frame.columns:
                s = frame[col]
                n_nonmissing = int(s.notna().sum())
                n_unique = int(s.nunique(dropna=True))
                numeric = pd.to_numeric(s, errors="coerce")
                finite = numeric[np.isfinite(numeric)]
                is_numeric_like = bool(len(finite) and (numeric.notna().sum() / max(1, n_nonmissing)) >= 0.8)
                uncertainty_column_rows.append({
                    "member": member,
                    "phase": ident[0] if ident else "UNKNOWN",
                    "canonical_run_id": ident[1] if ident else None,
                    "column": str(col),
                    "n_rows": int(len(frame)),
                    "n_nonmissing": n_nonmissing,
                    "n_unique": n_unique,
                    "varies_within_file": bool(n_unique > 1),
                    "numeric_like": is_numeric_like,
                    "looks_uncertainty_related_by_name": bool(column_name_hint.search(str(col))),
                    "looks_interpolation_related_by_name": bool(interpolation_name_hint.search(str(col))),
                    "min_finite": float(finite.min()) if len(finite) else np.nan,
                    "max_finite": float(finite.max()) if len(finite) else np.nan,
                    "std_finite": float(finite.std(ddof=0)) if len(finite) else np.nan,
                    "zero_fraction_of_finite": float((finite == 0).mean()) if len(finite) else np.nan,
                    "negative_fraction_of_finite": float((finite < 0).mean()) if len(finite) else np.nan,
                    "missing_fraction": float(s.isna().mean()) if len(s) else np.nan,
                    "sample_values_if_small_cardinality": json.dumps(s.dropna().astype(str).unique()[:12].tolist(), ensure_ascii=False) if n_unique <= 12 else None,
                })
        except Exception as exc:
            uncertainty_errors.append({"archive": UNC_KEY, "member": member, "error": repr(exc)})

uncertainty_columns_df = pd.DataFrame(uncertainty_column_rows)
uncertainty_files_df = pd.DataFrame(uncertainty_file_rows)
uncertainty_errors_df = pd.DataFrame(uncertainty_errors)
uncertainty_columns_df.to_csv(WORK_DIR / "uncertainty_column_diagnostics.csv", index=False)
uncertainty_files_df.to_csv(WORK_DIR / "uncertainty_file_diagnostics.csv", index=False)
uncertainty_errors_df.to_csv(WORK_DIR / "uncertainty_read_errors.csv", index=False)

print("Uncertainty files scanned:", len(uncertainty_files_df))
print("Uncertainty read errors:", len(uncertainty_errors_df))
print("Column-level rows:", len(uncertainty_columns_df))
print("\nColumns whose names suggest uncertainty/interpolation:")
display(uncertainty_columns_df[uncertainty_columns_df["looks_uncertainty_related_by_name"]].head(100))
print("\nPotentially time-varying numeric columns (examples only):")
dynamic_cols = uncertainty_columns_df[(uncertainty_columns_df["numeric_like"]) & (uncertainty_columns_df["varies_within_file"])]
display(dynamic_cols[["member", "column", "n_unique", "min_finite", "max_finite", "std_finite", "zero_fraction_of_finite", "missing_fraction"]].head(100))

sensor_interpolation_rows = []
sensor_interpolation_errors = []
with zipfile.ZipFile(local_files[SENSOR_KEY]) as zf:
    sensor_csvs = [n for n in zf.namelist() if n.lower().endswith(".csv") and not n.endswith("/")]
    for member in sensor_csvs:
        ident = identity_from_member(member)
        try:
            with zf.open(member) as stream:
                frame = pd.read_csv(stream, low_memory=False)
            for col in frame.columns:
                if interpolation_name_hint.search(str(col)):
                    tm, fm, mm, um = normalize_flag_values(frame[col])
                    sensor_interpolation_rows.append({
                        "member": member,
                        "phase": ident[0] if ident else "UNKNOWN",
                        "canonical_run_id": ident[1] if ident else None,
                        "column": str(col),
                        "n_rows": len(frame),
                        "true_count": int(tm.sum()),
                        "false_count": int(fm.sum()),
                        "missing_count": int(mm.sum()),
                        "unrecognized_count": int(um.sum()),
                        "semantics_review": "UNCONFIRMED: verify documentation/source",
                    })
        except Exception as exc:
            sensor_interpolation_errors.append({"member": member, "error": repr(exc)})

sensor_interp_df = pd.DataFrame(sensor_interpolation_rows)
sensor_interp_df.to_csv(WORK_DIR / "interpolation_flag_other_archives.csv", index=False)
pd.DataFrame(sensor_interpolation_errors).to_csv(WORK_DIR / "sensor_flag_scan_errors.csv", index=False)
print("\nInterpolation-like columns in sensor archive:", len(sensor_interp_df))
if not sensor_interp_df.empty:
    display(sensor_interp_df.head(50))
print("Saved uncertainty diagnostics in:", WORK_DIR)

## 6. Full YAML anomaly/perturbation registry

Every YAML file is read. The notebook records both `PerturbationMode.hasBeginning` (candidate perturbation onset) and the event-level `hasBeginning` (candidate observable anomaly onset). `00:00:00` is flagged for review, not automatically removed. YAML parse errors go into a separate file so they are not counted as anomaly records.

In [ ]:
def as_event_list(anomalies):
    if anomalies is None:
        return []
    if isinstance(anomalies, list):
        return anomalies
    if isinstance(anomalies, dict):
        event_keys = {str(k).lower() for k in anomalies}
        if event_keys.intersection({"id", "hasbeginning", "perturbationmode", "class", "type"}):
            return [anomalies]
        # Some YAML schemas may key event objects by event ID.
        return list(anomalies.values())
    return []


def as_mode_list(perturb):
    if perturb is None:
        return [None]
    if isinstance(perturb, list):
        return perturb if perturb else [None]
    return [perturb]


def get_ci(mapping, key, default=None):
    if not isinstance(mapping, dict):
        return default
    target = key.casefold()
    for k, v in mapping.items():
        if str(k).casefold() == target:
            return v
    return default


yaml_registry_rows = []
yaml_errors = []
yaml_files_without_events = []
with zipfile.ZipFile(local_files[LABEL_KEY]) as zf:
    yaml_members = [n for n in zf.namelist() if n.lower().endswith((".yaml", ".yml")) and not n.endswith("/")]
    print("YAML members:", len(yaml_members))
    for member in yaml_members:
        ident = identity_from_member(member)
        phase = ident[0] if ident else "UNKNOWN"
        run_id = ident[1] if ident else None
        exp_key = f"{phase}/{run_id}" if run_id else f"{phase}/{member}"
        try:
            data = yaml.safe_load(zf.read(member).decode("utf-8", errors="replace"))
            anomalies = get_ci(data, "anomalies", []) if isinstance(data, dict) else []
            events = as_event_list(anomalies)
            if not events:
                yaml_files_without_events.append({"yaml_file": member, "experiment_key": exp_key})
                continue

            for event_idx, event in enumerate(events):
                if not isinstance(event, dict):
                    continue
                anomaly_id = get_ci(event, "id", f"EVENT_INDEX_{event_idx}")
                anomaly_onset = get_ci(event, "hasBeginning")
                anomaly_end = get_ci(event, "hasEnd")
                perturb = get_ci(event, "PerturbationMode")
                modes = as_mode_list(perturb)
                for mode_idx, mode in enumerate(modes):
                    perturb_onset = get_ci(mode, "hasBeginning") if isinstance(mode, dict) else None
                    perturb_end = get_ci(mode, "hasEnd") if isinstance(mode, dict) else None
                    onset_str = str(perturb_onset).strip() if perturb_onset is not None else ""
                    yaml_registry_rows.append({
                        "experiment_key": exp_key,
                        "experiment_id": run_id,
                        "phase": phase,
                        "yaml_file": member,
                        "event_index": event_idx,
                        "anomaly_id": str(anomaly_id),
                        "anomaly_class_candidate": str(get_ci(event, "class", get_ci(event, "type", ""))),
                        "anomaly_onset": anomaly_onset,
                        "anomaly_end": anomaly_end,
                        "mode_index": mode_idx,
                        "n_modes_in_entry": len(modes),
                        "perturbation_onset": perturb_onset,
                        "perturbation_end": perturb_end,
                        "onset_is_zero_clock": onset_str == "00:00:00",
                        "onset_missing": perturb_onset is None or onset_str == "",
                        "review_status": "NOT_REVIEWED",
                    })
        except Exception as exc:
            yaml_errors.append({"yaml_file": member, "experiment_key": exp_key, "error": repr(exc)})

yaml_registry_df = pd.DataFrame(yaml_registry_rows)
yaml_errors_df = pd.DataFrame(yaml_errors)
yaml_no_events_df = pd.DataFrame(yaml_files_without_events)
yaml_registry_path = WORK_DIR / "pre_onset_registry_full.csv"
yaml_errors_path = WORK_DIR / "pre_onset_registry_errors.csv"
yaml_registry_df.to_csv(yaml_registry_path, index=False)
yaml_errors_df.to_csv(yaml_errors_path, index=False)
yaml_no_events_df.to_csv(WORK_DIR / "yaml_files_without_events.csv", index=False)

print("\n=== YAML audit summary ===")
print("YAML files scanned:", len(yaml_members))
print("YAML files with parse/structure errors:", len(yaml_errors_df))
print("YAML files without recognized anomaly entries:", len(yaml_no_events_df))
print("Extracted perturbation-mode records (includes missing onsets):", len(yaml_registry_df))
if not yaml_registry_df.empty:
    nonzero_mask = yaml_registry_df["perturbation_onset"].notna() & ~yaml_registry_df["onset_is_zero_clock"] & ~yaml_registry_df["onset_missing"]
    print("Non-zero perturbation-onset candidates:", int(nonzero_mask.sum()))
    print("Zero-clock onset candidates (review needed):", int(yaml_registry_df["onset_is_zero_clock"].sum()))
    print("Missing perturbation onset:", int(yaml_registry_df["onset_missing"].sum()))
    print("Unique experiment keys with non-zero onset:", int(yaml_registry_df.loc[nonzero_mask, "experiment_key"].nunique()))
    print("Records from multi-mode entries:", int((yaml_registry_df["n_modes_in_entry"] > 1).sum()))
    print("\nSample of non-zero-onset candidates:")
    display(yaml_registry_df.loc[nonzero_mask, ["experiment_key", "anomaly_id", "mode_index", "perturbation_onset", "anomaly_onset", "onset_is_zero_clock"]].head(10))
print("Saved:", yaml_registry_path)
print("Saved:", yaml_errors_path)
print("WARNING: Extracted onset values are candidates, not verified operation-relative seconds.")

## 7. Inspect every XML configuration and create time/calibration candidates

The XML scan indexes tags, attributes, and text suggesting time, perturbation, ramp, step, setpoint, fault, pressure, reflux/efflux, heat, or calibration. These are candidate fields for review, not automatic onset declarations. String hits such as `230.71` or `70000` are also only candidates.

In [ ]:
candidate_name = re.compile(r"time|start|end|perturb|fault|anomal|ramp|step|set.?point|efflux|reflux|heat|pressure|calibr", re.I)
config_candidate_rows = []
config_inventory_rows = []
config_parse_errors = []
calibration_text_matches = []

with zipfile.ZipFile(local_files[CONFIG_KEY]) as zf:
    xml_members = [n for n in zf.namelist() if n.lower().endswith(".xml") and not n.endswith("/")]
    print("XML configuration members:", len(xml_members))
    for member in xml_members:
        ident = identity_from_member(member)
        base = {
            "member": member,
            "phase_candidate": ident[0] if ident else "UNKNOWN",
            "canonical_run_id_candidate": ident[1] if ident else None,
        }
        try:
            root = ET.fromstring(zf.read(member))
            tags, attrs, texts = [], [], []
            def walk_xml(element, trail=""):
                tag = str(element.tag).split("}")[-1]
                current = f"{trail}/{tag}" if trail else tag
                text_value = (element.text or "").strip()
                attributes = {str(k): str(v) for k, v in element.attrib.items()}
                if current not in tags:
                    tags.append(current)
                if attributes:
                    attrs.append({"path": current, "attributes": attributes})
                if text_value:
                    texts.append({"path": current, "text": text_value[:500]})
                name_blob = " ".join([tag, current, *attributes.keys()])
                value_blob = " ".join([text_value, *attributes.values()])
                if candidate_name.search(name_blob) or candidate_name.search(value_blob):
                    config_candidate_rows.append({
                        **base,
                        "xml_path": current,
                        "tag": tag,
                        "text_value": text_value[:500],
                        "attributes_json": json.dumps(attributes, ensure_ascii=False),
                        "reviewed_semantics": "NOT_REVIEWED",
                        "unit": "UNKNOWN",
                        "time_origin": "UNKNOWN",
                        "is_perturbation_start": "NOT_REVIEWED",
                    })
                if any(token in value_blob for token in ("230.71", "70000", "70000.0")):
                    calibration_text_matches.append({
                        **base,
                        "xml_path": current,
                        "tag": tag,
                        "text_value": text_value[:500],
                        "attributes_json": json.dumps(attributes, ensure_ascii=False),
                        "candidate_only": True,
                    })
                for child in list(element):
                    walk_xml(child, current)
            walk_xml(root)
            config_inventory_rows.append({
                **base, "parse_ok": True, "root_tag": str(root.tag).split("}")[-1],
                "tag_count": len(tags), "tag_names_json": json.dumps(tags, ensure_ascii=False),
                "attributes_json": json.dumps(attrs, ensure_ascii=False),
                "text_samples_json": json.dumps(texts[:250], ensure_ascii=False), "parse_error": "",
            })
        except Exception as exc:
            err = repr(exc)
            config_parse_errors.append({"member": member, "error": err})
            config_inventory_rows.append({**base, "parse_ok": False, "parse_error": err})

config_candidates_df = pd.DataFrame(config_candidate_rows)
config_inventory_df = pd.DataFrame(config_inventory_rows)
calibration_candidates_df = pd.DataFrame(calibration_text_matches).drop_duplicates()
config_errors_df = pd.DataFrame(config_parse_errors)
config_candidates_df.to_csv(WORK_DIR / "simtrust_pre_onset_metadata_candidates.csv", index=False)
config_inventory_df.to_csv(WORK_DIR / "simulation_config_xml_inventory.csv", index=False)
calibration_candidates_df.to_csv(WORK_DIR / "simtrust_calibration_reference_candidates.csv", index=False)
config_errors_df.to_csv(WORK_DIR / "simtrust_config_xml_parse_errors.csv", index=False)

print("XML files inspected:", len(xml_members))
print("XML parse errors:", len(config_errors_df))
print("Candidate time/perturbation rows:", len(config_candidates_df))
print("Possible calibration text matches:", len(calibration_candidates_df))
print("Saved XML inventory:", WORK_DIR / "simulation_config_xml_inventory.csv")
print("Saved onset candidates:", WORK_DIR / "simtrust_pre_onset_metadata_candidates.csv")
print("Saved calibration candidates:", WORK_DIR / "simtrust_calibration_reference_candidates.csv")
print("\nSample candidate fields:")
display(config_candidates_df.head(20))

## 8. Human-review ledgers and shared-variable mapping template

This creates a review ledger by joining YAML-derived candidate perturbation records to structurally matching XML paths. The join is **only a candidate path join**. All rows start as `NOT_REVIEWED`; no time, unit, onset meaning, calibration reference, or real/sim pairing is automatically certified.

In [ ]:
# Candidate lookup from XML identities. Multiple XML members for one identity are retained as a joined string.
config_lookup_rows = []
for row in config_inventory_rows:
    if row.get("canonical_run_id_candidate"):
        config_lookup_rows.append({
            "experiment_key_candidate": f"{row.get('phase_candidate')}/{row.get('canonical_run_id_candidate')}",
            "configuration_member": row.get("member"),
        })
config_lookup_df = pd.DataFrame(config_lookup_rows)
if not config_lookup_df.empty:
    config_lookup_df = (config_lookup_df.groupby("experiment_key_candidate")["configuration_member"]
                        .agg(lambda x: ";".join(sorted(set(x.astype(str)))))
                        .reset_index())

review_columns = [
    "experiment_key", "experiment_id", "phase", "yaml_file", "anomaly_id", "event_index", "mode_index",
    "anomaly_onset_raw", "perturbation_onset_raw", "perturbation_end_raw", "configuration_member_candidate",
    "canonical_run_id_confirmed", "is_calibration_reference", "perturbation_family_reviewed",
    "perturbation_start_unit", "perturbation_start_time_origin", "operation_relative_onset_seconds",
    "paired_real_run_id", "time_alignment_evidence", "human_review_status", "reviewer_notes",
]
review_rows = []
for row in yaml_registry_rows:
    phase = row.get("phase") or "UNKNOWN"
    key = row.get("experiment_key")
    candidate_configs = ""
    if not config_lookup_df.empty and key:
        hit = config_lookup_df.loc[config_lookup_df["experiment_key_candidate"] == key, "configuration_member"]
        if len(hit):
            candidate_configs = hit.iloc[0]
    review_rows.append({
        "experiment_key": key,
        "experiment_id": row.get("experiment_id"),
        "phase": phase,
        "yaml_file": row.get("yaml_file"),
        "anomaly_id": row.get("anomaly_id"),
        "event_index": row.get("event_index"),
        "mode_index": row.get("mode_index"),
        "anomaly_onset_raw": row.get("anomaly_onset"),
        "perturbation_onset_raw": row.get("perturbation_onset"),
        "perturbation_end_raw": row.get("perturbation_end"),
        "configuration_member_candidate": candidate_configs,
        "canonical_run_id_confirmed": "",
        "is_calibration_reference": "NOT_REVIEWED",
        "perturbation_family_reviewed": "NOT_REVIEWED",
        "perturbation_start_unit": "UNKNOWN",
        "perturbation_start_time_origin": "UNKNOWN",
        "operation_relative_onset_seconds": np.nan,
        "paired_real_run_id": "",
        "time_alignment_evidence": "",
        "human_review_status": "NOT_REVIEWED",
        "reviewer_notes": "",
    })

review_df = pd.DataFrame(review_rows, columns=review_columns)
review_path = WORK_DIR / "pre_onset_review_registry.csv"
if not review_path.exists():
    review_df.to_csv(review_path, index=False)
    print("Created review ledger from current candidates:", review_path)
else:
    # Preserve an existing human-reviewed ledger; do not overwrite human annotations.
    print("Existing review ledger retained to preserve human annotations:", review_path)
    print("If you need a fresh blank/candidate ledger, rename or remove the existing file first.")

mapping_path = WORK_DIR / "shared_variable_mapping_template.csv"
mapping_columns = [
    "simulation_variable", "simulation_unit", "experimental_channel", "experimental_unit",
    "mapping_method_or_formula", "anomaly_ids_supported", "evidence_source",
    "include_in_primary_analysis", "review_status", "notes",
]
if not mapping_path.exists():
    pd.DataFrame(columns=mapping_columns).to_csv(mapping_path, index=False)
    print("Created variable mapping template:", mapping_path)
else:
    print("Existing variable mapping template retained:", mapping_path)

print("Review-ledger candidate rows:", len(review_df))
print("Sample review rows:")
display(review_df.head(12))

## 8B. YAML event identity and XML simulated-flag crosswalk audit (v4.2)

This section adds a conservative audit of event identity and the XML `anomaly_id` / `anomaly_id_simulated` association. It fixes two important hazards in earlier ad-hoc snippets:

- Canonical run keys are generated with `identity_from_member`, so YAML files named `<experiment>.yaml` and XML files stored as `<experiment>/config.xml` refer to the same `phase/chemical/operating_point/experiment` key.
- XML values are read within their `note` container (the documented `config/note/...` structure), preserving list positions. Empty list items are retained and flagged rather than silently removed. Files with mismatched node/list lengths are not paired automatically.

The output is still a **structural candidate audit**. A `TRUE` configuration flag is not, by itself, proof that an experimental time series is a valid scientific pair. Mixed flags or ambiguous XML structure require human review. No data are deleted and no labels are propagated.


In [ ]:
# ============================================================
# 8B. YAML object identity and XML simulation-status crosswalk
# ============================================================
from pathlib import PurePosixPath
from collections import Counter
import hashlib
import json
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
import yaml

V42_LABEL_KEY = "00_Batch_Distillation_Plant_M-202210_Timeseries_Label_Anomaly_Metadata.zip"
V42_CONFIG_KEY = "14_Batch_Distillation_Plant_M-202210_Timeseries_Simulation_Configuration.zip"


def _v42_archive_path(key):
    if "local_files" in globals() and key in local_files:
        path = Path(local_files[key])
    else:
        path = Path(WORK_DIR) / key
    if not path.exists():
        raise FileNotFoundError(f"Required archive not found: {path}")
    return path


def _v42_identity_key(member):
    """Canonical phase/chemical/operating-point/experiment key.

    Uses the notebook's tested parser instead of slicing the last path components,
    which drops the experiment ID for YAML files and misaligns YAML and XML keys.
    """
    identity = identity_from_member(member)
    if identity is None:
        return f"UNRESOLVED/{PurePosixPath(str(member)).as_posix()}"
    phase, canonical_run_id = identity
    return f"{phase}/{canonical_run_id}"


def _v42_get_ci(mapping, key, default=None):
    if not isinstance(mapping, dict):
        return default
    target = str(key).casefold()
    for k, value in mapping.items():
        if str(k).casefold() == target:
            return value
    return default


def _v42_event_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        event_keys = {str(k).casefold() for k in value}
        if event_keys.intersection({"id", "hasbeginning", "perturbationmode", "class", "type"}):
            return [value]
        return list(value.values())
    return []


def _v42_mode_list(value):
    if value is None:
        return [None]
    if isinstance(value, list):
        return value if value else [None]
    return [value]


def _v42_payload_hash(obj):
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest(), payload


def _v42_text(value):
    if value is None:
        return ""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
    return str(value).strip()


# A. Read every YAML event object and every perturbation-mode entry.
yaml_event_rows_v42 = []
yaml_mode_rows_v42 = []
yaml_errors_v42 = []
label_path_v42 = _v42_archive_path(V42_LABEL_KEY)

with zipfile.ZipFile(label_path_v42) as zf:
    yaml_members_v42 = [
        name for name in zf.namelist()
        if name.lower().endswith((".yaml", ".yml")) and not name.endswith("/")
    ]
    for member in yaml_members_v42:
        try:
            data = yaml.safe_load(zf.read(member).decode("utf-8", errors="replace"))
            anomalies = _v42_event_list(_v42_get_ci(data, "anomalies", []))
            run_key = _v42_identity_key(member)

            for event_idx, event in enumerate(anomalies):
                if not isinstance(event, dict):
                    continue

                event_hash, event_payload = _v42_payload_hash(event)
                anomaly_id = _v42_text(_v42_get_ci(event, "id", ""))
                anomaly_onset = _v42_text(_v42_get_ci(event, "hasBeginning", ""))
                anomaly_end = _v42_text(_v42_get_ci(event, "hasEnd", ""))
                anomaly_class = _v42_text(
                    _v42_get_ci(event, "class", _v42_get_ci(event, "type", ""))
                )

                yaml_event_rows_v42.append({
                    "yaml_file": member,
                    "experiment_key": run_key,
                    "event_index": event_idx,
                    "anomaly_id": anomaly_id,
                    "anomaly_class_candidate": anomaly_class,
                    "anomaly_onset": anomaly_onset,
                    "anomaly_end": anomaly_end,
                    "event_payload_sha256": event_hash,
                    "event_payload_json": event_payload,
                })

                perturb = _v42_get_ci(event, "PerturbationMode")
                modes = _v42_mode_list(perturb)
                for mode_idx, mode in enumerate(modes):
                    mode_hash, mode_payload = _v42_payload_hash(mode)
                    yaml_mode_rows_v42.append({
                        "yaml_file": member,
                        "experiment_key": run_key,
                        "event_index": event_idx,
                        "anomaly_id": anomaly_id,
                        "anomaly_onset": anomaly_onset,
                        "anomaly_end": anomaly_end,
                        "mode_index": mode_idx,
                        "n_modes_in_entry": len(modes),
                        "perturbation_onset": _v42_text(_v42_get_ci(mode, "hasBeginning", "")),
                        "perturbation_end": _v42_text(_v42_get_ci(mode, "hasEnd", "")),
                        "event_payload_sha256": event_hash,
                        "mode_payload_sha256": mode_hash,
                        "mode_payload_json": mode_payload,
                    })
        except Exception as exc:
            yaml_errors_v42.append({"yaml_file": member, "error": repr(exc)})

yaml_events_v42 = pd.DataFrame(yaml_event_rows_v42)
yaml_modes_v42 = pd.DataFrame(yaml_mode_rows_v42)
yaml_errors_df_v42 = pd.DataFrame(yaml_errors_v42)

# Exact full event-object repetitions within a YAML file.
if not yaml_events_v42.empty:
    exact_groups_v42 = (
        yaml_events_v42.groupby(["yaml_file", "event_payload_sha256"], dropna=False)
        .agg(
            row_count=("event_index", "size"),
            event_indices=("event_index", lambda s: ",".join(map(str, s))),
            anomaly_ids=("anomaly_id", lambda s: " | ".join(map(str, s))),
        )
        .reset_index()
    )
    exact_repeated_v42 = exact_groups_v42[exact_groups_v42["row_count"] > 1].copy()
else:
    exact_repeated_v42 = pd.DataFrame(columns=["yaml_file", "event_payload_sha256", "row_count"])

# Repeated signatures are *candidates*, not automatic instructions to delete rows.
mode_key_cols_v42 = [
    "yaml_file", "anomaly_id", "perturbation_onset", "perturbation_end",
    "anomaly_onset", "anomaly_end", "mode_index",
]
if not yaml_modes_v42.empty:
    key_groups_v42 = (
        yaml_modes_v42.groupby(mode_key_cols_v42, dropna=False)
        .agg(
            row_count=("event_payload_sha256", "size"),
            unique_payloads=("event_payload_sha256", "nunique"),
            event_indices=("event_index", lambda s: ",".join(map(str, s))),
        )
        .reset_index()
        .sort_values("row_count", ascending=False)
    )
    key_groups_v42["duplicate_interpretation"] = key_groups_v42.apply(
        lambda r: "SINGLE" if r["row_count"] == 1 else (
            "EXACT_DUPLICATE_FULL_EVENT" if r["unique_payloads"] == 1
            else "SHARED_KEY_DIFFERENT_PAYLOAD"
        ), axis=1
    )
    repeated_key_groups_v42 = key_groups_v42[key_groups_v42["row_count"] > 1].copy()
else:
    key_groups_v42 = pd.DataFrame()
    repeated_key_groups_v42 = pd.DataFrame()

# Save row-level and grouped audit files.
yaml_events_v42.drop(columns=["event_payload_json"], errors="ignore").to_csv(
    WORK_DIR / "yaml_event_identity_audit_v4_2.csv", index=False
)
yaml_modes_v42.drop(columns=["mode_payload_json"], errors="ignore").to_csv(
    WORK_DIR / "yaml_event_mode_audit_v4_2.csv", index=False
)
exact_repeated_v42.to_csv(
    WORK_DIR / "yaml_exact_duplicate_event_candidates_v4_2.csv", index=False
)
key_groups_v42.to_csv(
    WORK_DIR / "yaml_event_key_groups_v4_2.csv", index=False
)
yaml_errors_df_v42.to_csv(
    WORK_DIR / "yaml_event_identity_errors_v4_2.csv", index=False
)

print("=== YAML event identity audit v4.2 ===")
print("YAML files scanned:", len(yaml_members_v42))
print("YAML parse errors:", len(yaml_errors_df_v42))
print("Event objects extracted:", len(yaml_events_v42))
print("Event-mode rows extracted:", len(yaml_modes_v42))
print("Distinct event-file/hash groups:", len(yaml_events_v42.groupby(["yaml_file", "event_payload_sha256"])) if not yaml_events_v42.empty else 0)
print("Exact repeated full-event groups:", len(exact_repeated_v42))
print("Repeated key-signature groups:", len(repeated_key_groups_v42))
if not key_groups_v42.empty:
    print("Duplicate interpretation counts:")
    print(key_groups_v42["duplicate_interpretation"].value_counts().to_string())
    print("\nTop 15 repeated key-signature groups:")
    print(repeated_key_groups_v42.head(15).to_string(index=False))


# B. Parse anomaly IDs and simulated flags within each XML <note> container.
def _v42_local_tag(element):
    return str(element.tag).split("}")[-1].casefold()


def _v42_split_node_text(element):
    # Intentionally preserve empty tokens; dropping them independently can shift alignment.
    text = "" if element is None or element.text is None else str(element.text)
    return [part.strip() for part in text.split(",")]


def _v42_simulation_status(value):
    token = str(value).strip().lower()
    if token in {"true", "1", "1.0", "yes", "y", "t"}:
        return "TRUE"
    if token in {"false", "0", "0.0", "no", "n", "f"}:
        return "FALSE"
    return "UNKNOWN"


def _v42_extract_xml_pairs(root, member, experiment_key):
    pair_rows = []
    issues = []
    note_elements = [e for e in root.iter() if _v42_local_tag(e) == "note"]
    contexts = [(f"note_{i}", note) for i, note in enumerate(note_elements)]
    pairing_scope = "note_direct_children"

    # If the expected note structure is absent, only allow a fallback when there is
    # exactly one ID node and exactly one flag node in the whole document.
    if not contexts:
        all_ids = [e for e in root.iter() if _v42_local_tag(e) == "anomaly_id"]
        all_flags = [e for e in root.iter() if _v42_local_tag(e) == "anomaly_id_simulated"]
        if len(all_ids) == 1 and len(all_flags) == 1:
            contexts = [("root_fallback", root)]
            pairing_scope = "single_pair_root_fallback_review_required"
        else:
            issues.append(
                f"No note container; root fallback requires exactly one ID and one flag node "
                f"(found IDs={len(all_ids)}, flags={len(all_flags)})"
            )

    n_id_nodes = 0
    n_flag_nodes = 0
    n_id_tokens = 0
    n_flag_tokens = 0
    for context_name, context in contexts:
        # Public configs document config/note/anomaly_id and config/note/anomaly_id_simulated.
        ids_nodes = [e for e in list(context) if _v42_local_tag(e) == "anomaly_id"]
        flag_nodes = [e for e in list(context) if _v42_local_tag(e) == "anomaly_id_simulated"]

        # Root fallback allows its one matching pair even if not direct child due to namespace only.
        if context_name == "root_fallback":
            ids_nodes = [e for e in context.iter() if _v42_local_tag(e) == "anomaly_id"]
            flag_nodes = [e for e in context.iter() if _v42_local_tag(e) == "anomaly_id_simulated"]

        if not ids_nodes and not flag_nodes:
            continue

        n_id_nodes += len(ids_nodes)
        n_flag_nodes += len(flag_nodes)
        if len(ids_nodes) != len(flag_nodes):
            issues.append(
                f"{context_name}: node-count mismatch IDs={len(ids_nodes)} flags={len(flag_nodes)}"
            )
            continue

        for node_index, (id_node, flag_node) in enumerate(zip(ids_nodes, flag_nodes)):
            ids = _v42_split_node_text(id_node)
            flags = _v42_split_node_text(flag_node)
            n_id_tokens += len(ids)
            n_flag_tokens += len(flags)
            if len(ids) != len(flags):
                issues.append(
                    f"{context_name}, node_pair_{node_index}: token-count mismatch "
                    f"IDs={len(ids)} flags={len(flags)}"
                )
                continue

            for pos, (aid, flag) in enumerate(zip(ids, flags)):
                pair_rows.append({
                    "configuration_member": member,
                    "experiment_key": experiment_key,
                    "context": context_name,
                    "node_pair_index": node_index,
                    "list_position": pos,
                    "anomaly_id": aid,
                    "simulated_flag_raw": flag,
                    "simulated_status": _v42_simulation_status(flag),
                    "blank_anomaly_id": aid == "",
                    "blank_flag": flag == "",
                    "pairing_scope": pairing_scope,
                    "pairing_review_required": True,
                })

    has_tags = (n_id_nodes + n_flag_nodes) > 0
    list_lengths_match = bool(has_tags and not issues and len(pair_rows) > 0)
    return pair_rows, {
        "configuration_member": member,
        "experiment_key": experiment_key,
        "note_container_count": len(note_elements),
        "anomaly_id_node_count": n_id_nodes,
        "simulated_flag_node_count": n_flag_nodes,
        "anomaly_id_token_count": n_id_tokens,
        "simulated_flag_token_count": n_flag_tokens,
        "aligned_positions_extracted": len(pair_rows),
        "list_lengths_match": list_lengths_match if has_tags else None,
        "pairing_scope": pairing_scope,
        "issues": " | ".join(issues),
        "parse_error": "",
    }


config_path_v42 = _v42_archive_path(V42_CONFIG_KEY)
xml_pairs_rows_v42 = []
xml_summary_rows_v42 = []

with zipfile.ZipFile(config_path_v42) as zf:
    xml_members_v42 = [
        name for name in zf.namelist()
        if name.lower().endswith(".xml") and not name.endswith("/")
    ]
    for member in xml_members_v42:
        run_key = _v42_identity_key(member)
        try:
            root = ET.fromstring(zf.read(member))
            pair_rows, summary_row = _v42_extract_xml_pairs(root, member, run_key)
            xml_pairs_rows_v42.extend(pair_rows)
            xml_summary_rows_v42.append(summary_row)
        except Exception as exc:
            xml_summary_rows_v42.append({
                "configuration_member": member,
                "experiment_key": run_key,
                "list_lengths_match": False,
                "issues": "",
                "parse_error": repr(exc),
            })

xml_pairs_v42 = pd.DataFrame(xml_pairs_rows_v42)
xml_summary_v42 = pd.DataFrame(xml_summary_rows_v42)
xml_pairs_v42.to_csv(WORK_DIR / "xml_anomaly_flag_pairs_v4_2.csv", index=False)
xml_summary_v42.to_csv(WORK_DIR / "xml_list_alignment_audit_v4_2.csv", index=False)

print("\n=== XML list-alignment audit v4.2 ===")
print("XML files scanned:", len(xml_summary_v42))
print("XML parse errors:", int(xml_summary_v42.get("parse_error", pd.Series(dtype=str)).fillna("").ne("").sum()))
print("Files with matching ID/flag lists:", int(xml_summary_v42.get("list_lengths_match", pd.Series(dtype=bool)).eq(True).sum()))
print("Files with mismatched/ambiguous lists:", int(xml_summary_v42.get("list_lengths_match", pd.Series(dtype=bool)).eq(False).sum()))
print("Files with no usable ID/flag lists:", int(xml_summary_v42.get("list_lengths_match", pd.Series(dtype=object)).isna().sum()))
print("Aligned positions extracted:", len(xml_pairs_v42))
if not xml_pairs_v42.empty:
    print("TRUE positions:", int(xml_pairs_v42["simulated_status"].eq("TRUE").sum()))
    print("FALSE positions:", int(xml_pairs_v42["simulated_status"].eq("FALSE").sum()))
    print("UNKNOWN positions:", int(xml_pairs_v42["simulated_status"].eq("UNKNOWN").sum()))
    print("Blank anomaly-ID positions:", int(xml_pairs_v42["blank_anomaly_id"].sum()))
    print("\nSample XML ID/flag positions:")
    print(xml_pairs_v42[[
        "experiment_key", "context", "list_position", "anomaly_id",
        "simulated_flag_raw", "simulated_status", "pairing_scope"
    ]].head(25).to_string(index=False))


# C. Conservative YAML -> XML crosswalk, preserving mixed/unknown statuses.
def _v42_summarize_status(values):
    statuses = set(values)
    if statuses == {"TRUE"}:
        return "TRUE"
    if statuses == {"FALSE"}:
        return "FALSE"
    if statuses == {"UNKNOWN"}:
        return "UNKNOWN"
    if statuses == {"TRUE", "FALSE"}:
        return "MIXED_TRUE_FALSE"
    if statuses.intersection({"TRUE", "FALSE"}) and "UNKNOWN" in statuses:
        return "MIXED_WITH_UNKNOWN"
    return "UNKNOWN"


if not yaml_events_v42.empty and not xml_pairs_v42.empty:
    xml_nonblank_v42 = xml_pairs_v42[xml_pairs_v42["anomaly_id"].ne("")].copy()
    xml_by_event_v42 = (
        xml_nonblank_v42.groupby(["experiment_key", "anomaly_id"], dropna=False)["simulated_status"]
        .agg(_v42_summarize_status)
        .reset_index(name="xml_simulation_status")
    )
    yaml_event_crosswalk_v42 = yaml_events_v42.merge(
        xml_by_event_v42,
        on=["experiment_key", "anomaly_id"],
        how="left",
        validate="many_to_one",
    )
    yaml_event_crosswalk_v42["xml_simulation_status"] = (
        yaml_event_crosswalk_v42["xml_simulation_status"].fillna("NOT_FOUND_IN_XML")
    )
    yaml_event_crosswalk_v42["review_required"] = True
    yaml_event_crosswalk_v42.to_csv(
        WORK_DIR / "yaml_xml_anomaly_crosswalk_v4_2.csv", index=False
    )

    unique_crosswalk_v42 = (
        yaml_event_crosswalk_v42[["experiment_key", "anomaly_id", "xml_simulation_status"]]
        .drop_duplicates()
    )
    unique_crosswalk_v42.to_csv(
        WORK_DIR / "yaml_xml_anomaly_crosswalk_unique_v4_2.csv", index=False
    )

    print("\n=== YAML <-> XML crosswalk v4.2 ===")
    print("YAML event objects:", len(yaml_events_v42))
    print("Distinct YAML run/anomaly-ID pairs:", len(unique_crosswalk_v42))
    print("Distinct pairs with any XML status:", int(
        unique_crosswalk_v42["xml_simulation_status"].ne("NOT_FOUND_IN_XML").sum()
    ))
    print("\nXML status counts (distinct run/anomaly-ID pairs):")
    print(unique_crosswalk_v42["xml_simulation_status"].value_counts().to_string())
    print("\nSample crosswalk rows:")
    print(yaml_event_crosswalk_v42[[
        "experiment_key", "event_index", "anomaly_id",
        "anomaly_onset", "xml_simulation_status"
    ]].head(30).to_string(index=False))
else:
    yaml_event_crosswalk_v42 = pd.DataFrame()
    unique_crosswalk_v42 = pd.DataFrame()
    print("\nCrosswalk not created: YAML event objects or aligned XML pairs are empty.")


# D. Synthetic regression tests: key matching and conservative list alignment.
_test_yaml_path = "archive/Operation/chem_A/OP_01/experiment_001.yaml"
_test_xml_path = "archive/Operation/chem_A/OP_01/experiment_001/config.xml"
assert _v42_identity_key(_test_yaml_path) == _v42_identity_key(_test_xml_path), (
    _v42_identity_key(_test_yaml_path), _v42_identity_key(_test_xml_path)
)
_test_root = ET.fromstring(
    "<config><note><anomaly_id>A001, A002</anomaly_id>"
    "<anomaly_id_simulated>True, False</anomaly_id_simulated></note></config>"
)
_test_pairs, _test_summary = _v42_extract_xml_pairs(
    _test_root, _test_xml_path, _v42_identity_key(_test_xml_path)
)
assert len(_test_pairs) == 2
assert [_x["simulated_status"] for _x in _test_pairs] == ["TRUE", "FALSE"]
assert _test_summary["list_lengths_match"] is True
_test_bad_root = ET.fromstring(
    "<config><note><anomaly_id>A001, A002</anomaly_id>"
    "<anomaly_id_simulated>True</anomaly_id_simulated></note></config>"
)
_test_bad_pairs, _test_bad_summary = _v42_extract_xml_pairs(
    _test_bad_root, _test_xml_path, _v42_identity_key(_test_xml_path)
)
assert len(_test_bad_pairs) == 0
assert _test_bad_summary["list_lengths_match"] is False
print("\nSynthetic v4.2 checks passed: YAML/XML keys, positional pairing, and mismatch rejection.")


## 9. Compact summary for sharing

Creates a human-readable `compact_audit_summary.txt` and machine-readable `compact_audit_summary.json`. The summary records file counts and structural matches, not scientific validation.

In [ ]:
def safe_int(value):
    try:
        return int(value)
    except Exception:
        return None

summary = {
    "record": {"id": RECORD_ID, "version": EXPECTED_VERSION, "doi": RECORD_DOI},
    "working_directory": str(WORK_DIR.resolve()),
    "archive_counts": archive_counts,
    "all_four_archive_shared_phase_run_keys": len(all_four_shared),
    "pairwise_shared_phase_run_counts": {
        f"{Path(a).name} <> {Path(b).name}": len(archive_index[a]["identities"] & archive_index[b]["identities"])
        for a, b in [(LABEL_KEY, SENSOR_KEY), (LABEL_KEY, UNC_KEY), (LABEL_KEY, CONFIG_KEY), (SENSOR_KEY, UNC_KEY)]
    },
    "interpolation_audit": {
        "files_scanned": len(flags_df),
        "read_errors": len(flag_errors_df),
        "files_without_flag": int((~flags_df["flag_column_present"]).sum()) if len(flags_df) else None,
        "recognized_true": true_total,
        "recognized_false": false_total,
        "missing_values": missing_total,
        "unrecognized_values": unknown_total,
        "fraction_true_among_recognized": true_total / recognized_total if recognized_total else None,
    },
    "yaml_registry": {
        "yaml_files_scanned": len(yaml_members),
        "yaml_errors": len(yaml_errors_df),
        "yaml_files_without_recognized_events": len(yaml_no_events_df),
        "candidate_records": len(yaml_registry_df),
        "nonzero_onset_candidates": int((yaml_registry_df["perturbation_onset"].notna() & ~yaml_registry_df["onset_is_zero_clock"] & ~yaml_registry_df["onset_missing"]).sum()) if len(yaml_registry_df) else 0,
        "zero_clock_candidates": int(yaml_registry_df["onset_is_zero_clock"].sum()) if len(yaml_registry_df) else 0,
        "missing_onset_candidates": int(yaml_registry_df["onset_missing"].sum()) if len(yaml_registry_df) else 0,
        "unique_experiment_keys_with_nonzero_onset": int(yaml_registry_df.loc[yaml_registry_df["perturbation_onset"].notna() & ~yaml_registry_df["onset_is_zero_clock"] & ~yaml_registry_df["onset_missing"], "experiment_key"].nunique()) if len(yaml_registry_df) else 0,
    },
    "xml_configuration": {
        "xml_files_scanned": len(xml_members),
        "parse_errors": len(config_errors_df),
        "onset_candidate_rows": len(config_candidates_df),
        "calibration_text_candidate_rows": len(calibration_candidates_df),
    },
    "scientific_status": "STRUCTURAL AUDIT ONLY; pairing, variable mapping, time units/origin, uncertainty meaning, and label semantics require human review.",
}
summary_json_path = WORK_DIR / "compact_audit_summary.json"
summary_txt_path = WORK_DIR / "compact_audit_summary.txt"
summary_json_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

lines = [
    "SimTrust-Pre — Stage 1 compact audit summary",
    f"Record: {RECORD_ID} | version {EXPECTED_VERSION} | DOI {RECORD_DOI}",
    f"Working directory: {WORK_DIR.resolve()}",
    "",
    "Archive counts:",
]
for key, stats in archive_counts.items():
    lines.append(f"- {key}: {json.dumps(stats, ensure_ascii=False, sort_keys=True)}")
lines.extend([
    "", f"All-four-archive shared phase/run keys: {len(all_four_shared)}", "",
    "Interpolation flag audit:",
    f"- Files scanned: {len(flags_df)}; read errors: {len(flag_errors_df)}",
    f"- Files without Data_interpolated: {int((~flags_df['flag_column_present']).sum()) if len(flags_df) else 'N/A'}",
    f"- Recognized True: {true_total}; recognized False: {false_total}; missing: {missing_total}; unrecognized: {unknown_total}",
    f"- True fraction among recognized flags: {true_total / recognized_total:.6f}" if recognized_total else "- True fraction among recognized flags: UNDEFINED",
    "", "YAML perturbation-time candidates:",
    f"- YAML files scanned: {len(yaml_members)}; errors: {len(yaml_errors_df)}; no recognized events: {len(yaml_no_events_df)}",
    f"- Candidate perturbation records: {len(yaml_registry_df)}",
    f"- Nonzero onset candidates: {summary['yaml_registry']['nonzero_onset_candidates']}; zero-clock: {summary['yaml_registry']['zero_clock_candidates']}; missing: {summary['yaml_registry']['missing_onset_candidates']}",
    f"- Unique experiment keys with nonzero onset candidates: {summary['yaml_registry']['unique_experiment_keys_with_nonzero_onset']}",
    "", "XML configuration audit:",
    f"- XML files scanned: {len(xml_members)}; parse errors: {len(config_errors_df)}",
    f"- Time/perturbation candidate rows: {len(config_candidates_df)}; calibration text candidate rows: {len(calibration_candidates_df)}",
    "", "Interpretation warning:",
    "Structural candidates do not prove physical pairing. Review perturbation time units/time origin, map to operation-relative seconds, verify sensor-simulation identities and units, confirm uncertainty semantics, and define label classes before scoring/training.",
])
summary_txt_path.write_text("\n".join(lines), encoding="utf-8")
print("Saved:", summary_txt_path)
print("Saved:", summary_json_path)
print("\n".join(lines))

## 10. SimTrust-Pre discrepancy functions (research prototype)

This function block is intentionally independent of the audit and does not load or fit a detector. Inputs must already have a verified channel mapping, same physical units, same operation-relative time coordinate, and documented uncertainty semantics. Timestamp matching is exact; no hidden row-position pairing or interpolation occurs.

The ratio `q = |real - sim| / u` and the fraction `q <= kappa` are descriptive unless the source documentation justifies interpreting `u` as a physical tolerance/coverage bound. A zero or non-positive denominator is excluded, never converted to a tiny positive number.

In [ ]:
from typing import Any, Optional


def _coerce_boolean_series(series: pd.Series, *, label: str = "boolean flag") -> pd.Series:
    """Parse a documented boolean/0-1 flag; missing becomes NA, unknown tokens raise."""
    if not isinstance(series, pd.Series):
        raise TypeError(f"{label} must be a pandas Series")
    if series.index.has_duplicates:
        raise ValueError(f"{label} has duplicate time indices")
    true_values = {"true", "1", "yes", "y", "t"}
    false_values = {"false", "0", "no", "n", "f"}
    out = []
    for value in series.tolist():
        if pd.isna(value):
            out.append(pd.NA)
        elif isinstance(value, (bool, np.bool_)):
            out.append(bool(value))
        elif isinstance(value, (int, np.integer, float, np.floating)):
            if float(value) == 1.0:
                out.append(True)
            elif float(value) == 0.0:
                out.append(False)
            else:
                raise ValueError(f"{label} contains non-binary numeric value {value!r}")
        else:
            token = str(value).strip().lower()
            if token in true_values:
                out.append(True)
            elif token in false_values:
                out.append(False)
            elif token in {"", "nan", "none", "null"}:
                out.append(pd.NA)
            else:
                raise ValueError(f"{label} contains unrecognized value {value!r}; review its schema")
    return pd.Series(pd.array(out, dtype="boolean"), index=series.index, name=series.name)


def _window_metrics(
    real: pd.Series,
    sim: pd.Series,
    u: pd.Series,
    non_interpolated: Optional[pd.Series] = None,
    *,
    kappa: float,
    min_valid_points: int,
) -> dict[str, Any]:
    for label, series in (("real", real), ("sim", sim), ("uncertainty", u)):
        if not isinstance(series, pd.Series):
            raise TypeError(f"{label} must be a pandas Series")
        if series.index.has_duplicates:
            raise ValueError(f"{label} has duplicate time indices")
    pieces = [real.rename("real"), sim.rename("sim"), u.rename("u")]
    if non_interpolated is not None:
        pieces.append(_coerce_boolean_series(non_interpolated, label="non-interpolated mask").rename("non_interp"))
    frame = pd.concat(pieces, axis=1, join="inner")
    if frame.empty:
        raise ValueError("No exact shared timestamps; establish a documented common time grid first.")
    for col in ("real", "sim", "u"):
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
    numeric_finite = np.isfinite(frame[["real", "sim", "u"]]).all(axis=1)
    positive_u = frame["u"] > 0
    measured = (frame["non_interp"].fillna(False).astype(bool)
                if "non_interp" in frame.columns
                else pd.Series(True, index=frame.index))
    valid = numeric_finite & positive_u & measured
    n_valid = int(valid.sum())
    if n_valid < min_valid_points:
        raise ValueError(f"Only {n_valid} valid measured aligned points; minimum is {min_valid_points}.")
    f = frame.loc[valid]
    residual = f["real"] - f["sim"]
    q = residual.abs() / f["u"]
    return {
        "n_aligned": int(len(frame)),
        "n_valid": n_valid,
        "valid_fraction_of_aligned": float(valid.mean()),
        "interpolated_or_unknown_excluded_fraction": float((~measured).mean()),
        "raw_mae": float(residual.abs().mean()),
        "raw_rmse": float(np.sqrt(np.mean(np.square(residual)))),
        "mean_real_minus_sim_bias": float(residual.mean()),
        "median_q": float(q.median()),
        "p95_q": float(q.quantile(0.95)),
        "fraction_q_le_kappa": float((q <= kappa).mean()),
        "fraction_q_gt_kappa": float((q > kappa).mean()),
        "kappa": float(kappa),
    }


def simtrust_pre_window_metrics(
    real: pd.Series,
    sim: pd.Series,
    uncertainty: pd.Series,
    *,
    perturbation_start_s: float,
    window_s: float,
    kappa: float = 1.0,
    min_valid_points: int = 20,
    non_interpolated: Optional[pd.Series] = None,
    event_order: int = 1,
    previous_recovery_end_s: Optional[float] = None,
) -> dict[str, Any]:
    """Score early-prefix and immediate pre-onset windows plus a disjoint offset control.

    All indices must be verified elapsed seconds since the start of the Operation phase.
    Main windows: early [0,W), bias-fit [t-2W,t-W), pre-onset scoring [t-W,t).
    Primary analysis uses the first perturbation. Later events require reviewed recovery evidence.
    """
    if not np.isfinite(perturbation_start_s) or not np.isfinite(window_s) or window_s <= 0:
        raise ValueError("perturbation_start_s must be finite and window_s must be > 0")
    if not np.isfinite(kappa) or kappa <= 0:
        raise ValueError("kappa must be finite and > 0")
    if not isinstance(event_order, int) or event_order < 1:
        raise ValueError("event_order must be an integer >= 1")
    for label, series in (("real", real), ("sim", sim), ("uncertainty", uncertainty)):
        if not isinstance(series, pd.Series):
            raise TypeError(f"{label} must be a pandas Series")
        if series.index.has_duplicates:
            raise ValueError(f"{label} has duplicate time-index values")
        try:
            t = np.asarray(series.index, dtype=float)
        except Exception as exc:
            raise ValueError(f"{label} index must be numeric operation-relative seconds") from exc
        if not np.isfinite(t).all():
            raise ValueError(f"{label} index contains non-finite values")
    if non_interpolated is not None:
        _coerce_boolean_series(non_interpolated, label="non-interpolated mask")
    if perturbation_start_s < 2 * window_s:
        raise ValueError("Need at least 2W seconds before onset for disjoint bias-fit and pre-onset windows.")
    bias_start = perturbation_start_s - 2 * window_s
    score_start = perturbation_start_s - window_s
    if event_order > 1:
        if previous_recovery_end_s is None or not np.isfinite(previous_recovery_end_s):
            raise ValueError("Later perturbations require independently reviewed previous_recovery_end_s.")
        if previous_recovery_end_s > bias_start:
            raise ValueError("Previous event recovery ends after the bias-fit window starts; pre-onset status is not defensible.")

    def time_slice(series, start, end):
        t = np.asarray(series.index, dtype=float)
        return series.loc[(t >= start) & (t < end)]
    def mask_slice(start, end):
        return None if non_interpolated is None else time_slice(non_interpolated, start, end)

    early = (time_slice(real, 0.0, window_s), time_slice(sim, 0.0, window_s), time_slice(uncertainty, 0.0, window_s), mask_slice(0.0, window_s))
    pre = (time_slice(real, score_start, perturbation_start_s), time_slice(sim, score_start, perturbation_start_s), time_slice(uncertainty, score_start, perturbation_start_s), mask_slice(score_start, perturbation_start_s))
    bias_window = (time_slice(real, bias_start, score_start), time_slice(sim, bias_start, score_start), time_slice(uncertainty, bias_start, score_start), mask_slice(bias_start, score_start))

    result = {
        "window_s": float(window_s),
        "perturbation_start_s": float(perturbation_start_s),
        "event_order": event_order,
        "first_event_primary_eligible": bool(event_order == 1),
        "early_prefix": _window_metrics(*early[:3], early[3], kappa=kappa, min_valid_points=min_valid_points),
        "immediate_pre_onset": _window_metrics(*pre[:3], pre[3], kappa=kappa, min_valid_points=min_valid_points),
        "bias_fit_window": _window_metrics(*bias_window[:3], bias_window[3], kappa=kappa, min_valid_points=min_valid_points),
    }
    overlap_start, overlap_end = max(0.0, bias_start), min(window_s, score_start)
    overlap_duration = max(0.0, overlap_end - overlap_start)
    result["early_overlaps_bias_window"] = bool(overlap_duration > 0)
    result["early_bias_overlap_seconds"] = float(overlap_duration)

    bias_real, bias_sim, bias_u, bias_mask = bias_window
    fit_parts = [bias_real.rename("real"), bias_sim.rename("sim"), bias_u.rename("u")]
    if bias_mask is not None:
        fit_parts.append(_coerce_boolean_series(bias_mask, label="bias-fit non-interpolated mask").rename("non_interp"))
    fit_df = pd.concat(fit_parts, axis=1, join="inner")
    for col in ("real", "sim", "u"):
        fit_df[col] = pd.to_numeric(fit_df[col], errors="coerce")
    fit_valid = np.isfinite(fit_df[["real", "sim", "u"]]).all(axis=1) & (fit_df["u"] > 0)
    if "non_interp" in fit_df.columns:
        fit_valid &= fit_df["non_interp"].fillna(False).astype(bool)
    fit_df = fit_df.loc[fit_valid]
    if len(fit_df) < min_valid_points:
        raise ValueError(f"Only {len(fit_df)} valid points available for bias fit; minimum is {min_valid_points}.")
    bias = float((fit_df["real"] - fit_df["sim"]).mean())
    corrected_sim = pre[1] + bias
    result["offset_control"] = {
        "bias_fit_mean_real_minus_sim": bias,
        "bias_fit_n_valid": int(len(fit_df)),
        "scored_on_separate_immediate_pre_onset_window": True,
        "metrics_after_bias_correction": _window_metrics(pre[0], corrected_sim, pre[2], pre[3], kappa=kappa, min_valid_points=min_valid_points),
    }
    result["interpretation_warning"] = (
        "q and q<=kappa are descriptive unless uncertainty semantics and kappa are justified. "
        "Time boundaries require a verified operation-relative mapping. If early_overlaps_bias_window=True, "
        "the early-prefix metric overlaps the offset-fit interval."
    )
    return result

## 11. Offline self-tests (synthetic data only)

These tests verify code behavior and path parsing only. They are **not** dataset findings or evidence that the scientific hypothesis holds.

In [ ]:
# Test nullable flag parsing and rejection of unrecognized tokens.
_test_flags = pd.Series([True, False, "1", "0", None, "yes", "no"])
_test_parsed = _coerce_boolean_series(_test_flags, label="test flag")
assert list(_test_parsed.iloc[:4].astype(bool)) == [True, False, True, False]
assert pd.isna(_test_parsed.iloc[4])
try:
    _coerce_boolean_series(pd.Series(["maybe"]), label="bad test flag")
except ValueError:
    pass
else:
    raise AssertionError("Unknown boolean token was not rejected")

# Test score function with an intentionally simple signal.
_test_t = np.arange(0, 700, dtype=float)
_test_sim = pd.Series(20 + 0.01 * _test_t, index=_test_t)
_test_real = pd.Series(20 + 0.01 * _test_t + 0.2, index=_test_t)
_test_u = pd.Series(0.25, index=_test_t)
_test_measured = pd.Series(True, index=_test_t)
_test_measured.loc[[100.0, 101.0]] = False
_test = simtrust_pre_window_metrics(
    _test_real, _test_sim, _test_u,
    perturbation_start_s=300, window_s=60, kappa=1.0,
    min_valid_points=20, non_interpolated=_test_measured,
)
assert _test["early_prefix"]["n_valid"] > 0
assert _test["immediate_pre_onset"]["n_valid"] > 0
assert _test["offset_control"]["scored_on_separate_immediate_pre_onset_window"] is True
assert _test["early_overlaps_bias_window"] is False

_test_overlap = simtrust_pre_window_metrics(
    _test_real, _test_sim, _test_u,
    perturbation_start_s=150, window_s=60, kappa=1.0,
    min_valid_points=20, non_interpolated=_test_measured,
)
assert _test_overlap["early_overlaps_bias_window"] is True
assert _test_overlap["early_bias_overlap_seconds"] > 0

try:
    simtrust_pre_window_metrics(
        _test_real, _test_sim, _test_u,
        perturbation_start_s=300, window_s=60,
        event_order=2, min_valid_points=20,
    )
except ValueError:
    pass
else:
    raise AssertionError("Later event without recovery evidence should be rejected")

print("All offline synthetic self-tests passed. No dataset-level score was computed.")

## 12. Feasibility gate and pre-analysis protocol

Do **not** train a detector or claim dataset performance until these items are manually resolved:

- [ ] Exact calibration-reference experiment identified from paper/configuration evidence; report sensitivity with and without it.
- [ ] For each eligible event, verify `PerturbationMode.hasBeginning`, units, time origin, operation phase, and mapping to operation-relative seconds.
- [ ] Confirm which anomaly families are simulated. Do not include unsupported types in simulated-fault training claims.
- [ ] Confirm the meaning of `Data_interpolated` and the policy for false, true, missing, and unrecognized values.
- [ ] Confirm uncertainty-column semantics, units, zero/missing handling, and the interpretation (if any) of `q <= kappa`.
- [ ] Manually verify physical variable mappings and common units; column-name similarity alone is insufficient.
- [ ] Define target labels and detector type (supervised/semi-supervised/unsupervised) before computing `Delta AP`.
- [ ] Keep whole experiments and paired simulation cases in the same split. Test runs must not set thresholds, weights, features, or hyperparameters.
- [ ] Compute run-level utility only on development/validation data under fixed model/budget/split rules. Treat it as context-dependent, not an intrinsic property of a simulated run.
- [ ] If fewer than about 15 independent, valid pairs remain, label utility-prediction analysis exploratory and prioritize a descriptive benchmark.

### Proposed hypotheses (pre-register before looking at outcomes)

- **H1 — Ranking:** Does pre-onset uncertainty-scaled discrepancy rank the utility of simulated trajectories better than raw residual baselines and operating-condition baselines?
- **H2 — Weighting/selection:** Does a pre-onset fidelity-based selection/weighting policy improve real-validation performance over real-only, uniform simulation addition, raw-error weighting, and shuffled-weight controls?
- **Ablations, only if data allow:** early-prefix versus immediate-pre-onset windows; time-varying versus sensor-constant uncertainty; residual drift; a physically justified cross-sensor relationship.

Use a simple rank-based analysis first. With at most a few dozen events, avoid high-parameter prediction models and extensive post-hoc feature selection. A null/negative result is reportable when the protocol and limitations are transparent.

## 13. Optional: download the large simulation time-series ZIP

**Leave this disabled until the feasibility gate is reviewed.** It may be around 2.1 GB compressed. Changing the flag downloads the file and verifies size/checksum, but does not extract it or train a model.

In [ ]:
DOWNLOAD_SIMULATION = False  # Keep False until the small-archive audit and mapping review are complete.

if DOWNLOAD_SIMULATION:
    if SIM_KEY not in file_by_key:
        raise KeyError(f"The pinned record did not list the expected simulation archive: {SIM_KEY}")
    sim_path = download_file(SIM_KEY)
    print("Simulation archive downloaded and verified:", sim_path)
    print(f"Compressed size: {sim_path.stat().st_size / 1_000_000_000:.3f} GB")
    with zipfile.ZipFile(sim_path) as zf:
        sim_members = [n for n in zf.namelist() if n and not n.endswith("/")]
    print("Archive entries:", len(sim_members))
    for name in sim_members[:40]:
        print(" -", name)
else:
    print("Skipped large simulation archive. Finish the metadata and mapping review first.")

## 14. Expected output files

The notebook writes files beneath `WORK_DIR` (in Colab: `/content/simtrust_pre_audit/`). The v4.2 event/XML audit adds:

- `yaml_event_identity_audit_v4_2.csv`
- `yaml_event_mode_audit_v4_2.csv`
- `yaml_exact_duplicate_event_candidates_v4_2.csv`
- `yaml_event_key_groups_v4_2.csv`
- `yaml_event_identity_errors_v4_2.csv`
- `xml_anomaly_flag_pairs_v4_2.csv`
- `xml_list_alignment_audit_v4_2.csv`
- `yaml_xml_anomaly_crosswalk_v4_2.csv`
- `yaml_xml_anomaly_crosswalk_unique_v4_2.csv`

These files preserve ambiguous and mixed cases for human review. They do not prove physical pairing and do not trigger label propagation or model training.

The earlier Stage 1 outputs remain available. The bundle cell below includes these new audit outputs if this section is run before bundling.


In [ ]:
# Optional convenience: bundle the generated audit outputs into one ZIP for sharing.
# This cell does not run the audit again. Run it after the preceding cells complete.
import shutil
from pathlib import Path

bundle_base = WORK_DIR.parent / "simtrust_pre_audit_v4_2_outputs"
bundle_path = Path(shutil.make_archive(str(bundle_base), "zip", root_dir=WORK_DIR))
print("Audit output bundle:", bundle_path)
print(f"Bundle size: {bundle_path.stat().st_size / 1024:.1f} KB")

try:
    from google.colab import files
    files.download(str(bundle_path))
except ImportError:
    print("Google Colab download helper unavailable; bundle saved at:", bundle_path)
except Exception as exc:
    print("Automatic download could not be started:", repr(exc))
    print("Bundle remains saved at:", bundle_path)
